# `zarr` for Rapid Access to Large Image Stores

Today, we'll step through an example for creating a `zarr` store from an existing image dataset. This allows us to store many small images in a larger file size.

## Bring in CIFAR-10 dataset

In this example we're using the CIFAR-10 dataset because it's readily available, but you can swap in any datset of your preference.

In [2]:
import torchvision
import torchvision.transforms as transforms
import os

In [3]:
os.makedirs("./data", exist_ok=True)

We'll download the train/test splits seperately so we can save them as different zarr stores. This allows us to access them at different parts of our deep learning pipeline. CIFAR-10 arrives as PIL image which we convert to tensor of floats with the shape [C,H,W]. This allows us to store this more useful transformation in our `zarr` store.

In [4]:
transform = transforms.ToTensor()  

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True,  download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

print(f"Train images : {len(train_dataset):,}")   
print(f"Test  images : {len(test_dataset):,}")


00%|██████████| 170498071/170498071 [00:04<00:00, 38527918.53it/s]

Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Train images : 50,000
Test  images : 10,000


## Write the dataset to a `zarr` store

A `zarr` store behaves as a directory on disk for any arrays (in our case image arrays) that you store inside. It's especially powerful for rapid access to multi-dimensional arrays stored within it.

In [5]:
import zarr
import numpy as np
from tqdm import tqdm

We write a helper function here to initialize a zarr store and move our dataset into it. We do this by moving data into chunks and then writing those chunks. This is important for saving `zarr` stores efficiently. Every `zarr` write involves compressing data and writing that compressed file to disk. In this example, if you would write one image at a time to your store, you would trigger that overhead 50,000 times. By buffering, you do it only once per chunk, or 50 times.

In [6]:
def dataset_to_zarr(input_dataset, zarr_path, chunk_size=1000):
    """
    Write an image dataset (in an array of images) to a `zarr` store.
    """
    
    # Get image shape
    n = len(input_dataset)
    sample_img,_ = input_dataset[0]
    img_shape = tuple(sample_img.shape)

    # Open and initialize stores for images and labels
    store = zarr.open(zarr_path, mode="w")

    images = store.zeros(
        "images",
        shape = (n, *img_shape),
        chunks = (chunk_size, *img_shape),
        dtype = "float32",
    )

    labels = store.zeros(
        "labels",
        shape = (n,),
        chunks = (chunk_size,),
        dtype = "int64",
    )

    # For speed, fill arrays into chunks and then write chunks to zarr
    img_buf = np.empty((chunk_size, *img_shape), dtype="float32")
    lbl_buf = np.empty((chunk_size,), dtype="int64")

    buf_idx = 0
    arr_idx = 0

    for img_tensor, label in tqdm(input_dataset, desc=f"Writing {zarr_path}"):
        img_buf[buf_idx] = img_tensor.numpy()
        lbl_buf[buf_idx] = int(label)
        buf_idx += 1

        if buf_idx == chunk_size:
            images[arr_idx:(arr_idx+chunk_size)] = img_buf
            labels[arr_idx:(arr_idx+chunk_size)] = lbl_buf
            arr_idx += chunk_size
            buf_idx = 0

    if buf_idx > 0:
        images[arr_idx:arr_idx+buf_idx] = img_buf[:buf_idx]
        labels[arr_idx:arr_idx+buf_idx] = lbl_buf[:buf_idx]

    print(f"\nStore written to {zarr_path}")
    print(store.tree())
    return store

Now we'll use this to write our train store and our test store.

In [7]:
dataset_to_zarr(train_dataset, "cifar10_train.zarr", chunk_size=1000)
dataset_to_zarr(test_dataset, "cifar10_test.zarr",  chunk_size=1000)


riting cifar10_train.zarr: 100%|██████████| 50000/50000 [00:08<00:00, 5937.21it/s] 


Store written to cifar10_train.zarr
/
 ├── images (50000, 3, 32, 32) float32
 └── labels (50000,) int64


Writing cifar10_test.zarr: 100%|██████████| 10000/10000 [00:02<00:00, 4712.85it/s]


Store written to cifar10_test.zarr
/
 ├── images (10000, 3, 32, 32) float32
 └── labels (10000,) int64


<zarr.hierarchy.Group '/'>

We can use the built in `tree` and `info` functions to get a quick overview of our the components in our store, their shape, and their type.

In [8]:
store = zarr.open("cifar10_train.zarr", mode="r")
print(store.tree())

/
 ├── images (50000, 3, 32, 32) float32
 └── labels (50000,) int64


In [9]:
print(store["images"].info)

Name               : /images
Type               : zarr.core.Array
Data type          : float32
Shape              : (50000, 3, 32, 32)
Chunk shape        : (1000, 3, 32, 32)
Order              : C
Read-only          : True
Compressor         : Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0)
Store type         : zarr.storage.DirectoryStore
No. bytes          : 614400000 (585.9M)
No. bytes stored   : 495308514 (472.4M)
Storage ratio      : 1.2
Chunks initialized : 50/50



**How would this effect your inode count?** Under the hood, `zarr` took the dataset of 50,000 images and stitched them together into 50 chunks. We controlled this size by specifying 1000 chunks in our helper function, which divided the entire dataset into chunks with 1000 granules each (i.e. 50,000/1000 = 50 chunks). `zarr` still needs to store some metadata, so this collapses 50,000 files into ~55 files on disk, taking up 55 inodes instead of 50,000.

## How does this interact with PyTorch Datasets and Dataloaders?

In [10]:
import torch
from torch.utils.data import Dataset

Using PyTorch's Dataset class, we'll create a custom dataset that is `zarr` enabled, and pulls images directly from the `zarr` store that we just created.

In [11]:
class ZarrImageDataset(Dataset):
    """
    A `zarr` enabled pytorch dataset.

    The store is expected to have two internal branches:
      - store["images"]: [N,C,H,W] float32 
      - store["labels"]: [N,] int64
    """

    def __init__(self, zarr_path, transform=None):
        self.store = zarr.open(zarr_path, mode="r") # Make sure this is in read mode, "r"
        self.images = self.store["images"]
        self.labels = self.store["labels"]
        self.transform = transform

    def __len__(self):
        return self.images.shape[0]

    def __getitem__(self, idx):
        # zarr indexing returns numpy array so convert to tensor
        image = torch.from_numpy(self.images[idx])
        label = int(self.labels[idx])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

We can check that this iterates as expected in a PyTorch Dataloader.

In [12]:
from torch.utils.data import DataLoader

First we load our `zarr` stores as PyTorch datasets using our function above.

In [13]:
train_ds = ZarrImageDataset("cifar10_train.zarr")
test_ds = ZarrImageDataset("cifar10_test.zarr")

Next we create our Dataloaders. You can still specify batch size, shuffle, workers, pin memory, etc.

In [14]:
train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

In [15]:
test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

We can check that this is working in the same way that we'd expect loading from individual files would.

In [16]:
images, labels = next(iter(train_loader))

In [17]:
print(f"Batch image shape: {images.shape}")
print(f"Batch label shape: {labels.shape}")

Batch image shape: torch.Size([64, 3, 32, 32])
Batch label shape: torch.Size([64])


This is working! We specified a batch size of 64, and our individual images have the shape [3,32,32] as expected.

## How does this effect my I/O?

Understanding how `zarr` chunks interact with DataLoader batches is the key to getting good I/O throughput and keeping your jobs speedy.

When you read `images[idx]` inside your Dataloder, `zarr` goes through the following steps:
- automatically calculates which chunk contains row `i`
- reads that entire chunk from disk (if not already cached)
- returns the single row you asked for

In our example, where th chunk size is 1000 and the Dataloader reads the images in index order, `zarr` reads one chunk (i.e. images 0–999), pull each row from it from the cache, then reads the next chunk. That's only 50 disk reads for 50,000 images, so this might speed up your Dataloader when compared to reading individual images. However, typically only test arrays are read into the loader in their index order. What about test data?

If you read in your data in a *random* index order (e.g. with `shuffle=True` for testing), each time `images[idx]` is called, this potentially triggers a new disk read if the index is in a chunk that hasn't been opened recently. This means that you might want to store your data so that it is already "pre-chunked" into batches or small multiples of batches for faster reads.

Let's take a look at the difference that this makes to our read times.

In [24]:
import time

In [25]:
def benchmark(zarr_path, chunk_size, batch_size, n_batches=50):
    """
    Re-write the store with a new chunk size, then profile loading times.
    """

    # Create `zarr` store with specified chunk_size
    src = zarr.open(zarr_path, mode="r")
    dest_path = f"/tmp/bench_chunk{chunk_size}.zarr"
    dest = zarr.open(dest_path, mode="w")
    
    dest.zeros("images",
                shape=src["images"].shape,
                chunks=(chunk_size, *src["images"].shape[1:]),
                dtype="float32")
    
    dest.zeros("labels",
                shape=src["labels"].shape,
                chunks=(chunk_size,),
                dtype="int64")
    
    dest["images"][:] = src["images"][:]
    dest["labels"][:] = src["labels"][:]

    # Load this into pytorch Dataset and Loader
    ds = ZarrImageDataset(dest_path)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2)

    # Time how long it takes to load n_batches
    t0 = time.perf_counter()
    for i, _ in enumerate(loader):
        if i >= n_batches:
            break
            
    elapsed = time.perf_counter()-t0
    
    imgs_per_sec = (n_batches * batch_size)/elapsed
    
    print(f"chunk={chunk_size:>5}  batch={batch_size} --> {imgs_per_sec:,.0f} img/s")

In [27]:
for cs in [64, 128, 256, 512, 1024, 2048]:
    benchmark("cifar10_train.zarr", chunk_size=cs, batch_size=64)

chunk=   64  batch=64 --> 4,304 img/s
chunk=  128  batch=64 --> 2,703 img/s
chunk=  256  batch=64 --> 1,691 img/s
chunk=  512  batch=64 --> 842 img/s
chunk= 1024  batch=64 --> 432 img/s
chunk= 2048  batch=64 --> 218 img/s
